### Exploring the speaker Data and Moving into Duck DB

Run this into a database, because having the entire 7 gb corpus in memory with pandas would be quite slow. 

Codebook here: https://stacks.stanford.edu/file/druid:md374tz9962/codebook_v4.pdf

In [1]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

RAW = Path('../data/raw')
DERIVED = Path('../data/derived')
DERIVED.mkdir(exist_ok=True)
pd.__version__


'3.1.1'

In [2]:
#explore one text file to get a sense of structure

first_map = pd.read_csv(RAW / 'speakermap' / '043_SpeakerMap.txt', sep='|', quoting=3, dtype=str)
display(first_map.head())
print(first_map.shape[0])
first_map['speech_id'].nunique()
print(first_map.columns)


,speakerid,speech_id,lastname,firstname,chamber,state,gender,party,district,nonvoting
0,43044451,430000002,HAMLIN,HANNIBAL,S,ME,M,R,NaN,voting
1,43049531,430000007,ANTHONY,HENRY,S,RI,M,R,NaN,voting
2,43049531,430000008,ANTHONY,HENRY,S,RI,M,R,NaN,voting
3,43047111,430000010,CONKLING,ROSCOE,S,NY,M,R,NaN,voting
4,43044451,430000011,HAMLIN,HANNIBAL,S,ME,M,R,NaN,voting


96115
Index(['speakerid', 'speech_id', 'lastname', 'firstname', 'chamber', 'state',
       'gender', 'party', 'district', 'nonvoting'],
      dtype='str')


Speaker file merge - create one parquet of these. 

In [3]:
con = duckdb.connect(str(DERIVED / 'speakermap.duckdb')) #make a new database 

con.execute(f'''
CREATE OR REPLACE TABLE speakermap AS 
SELECT CAST(speakerid AS BIGINT) AS speakerid,
       CAST(speech_id AS BIGINT) AS speech_id, -- match data formats for the speech files. 
       lastname, firstname, chamber, state, gender, party, district, nonvoting
FROM read_csv('{RAW}/speakermap/*_SpeakerMap.txt', delim='|', header=true, quote='', all_varchar=true) -- same format as above, just in duckdb instead of pandas. 
''')

con.sql('DESCRIBE speakermap').df()[['column_name', 'column_type']] #list the columns and the types of each one, from lecture. 

,column_name,column_type
0,speakerid,BIGINT
1,speech_id,BIGINT
2,lastname,VARCHAR
3,firstname,VARCHAR
4,chamber,VARCHAR
5,state,VARCHAR
6,gender,VARCHAR
7,party,VARCHAR
8,district,VARCHAR
9,nonvoting,VARCHAR


Files merged together properly into one duckdb object. Run a few checks to make sure it is all good, and then put into a parquet. 

In [4]:
display(
    con.sql('''
        SELECT COUNT(*) AS rows, COUNT(DISTINCT speech_id) AS speeches
        FROM speakermap
    ''').df()
)

con.sql('''
SELECT COUNT(*), party
FROM speakermap
GROUP BY party
''').df()

,rows,speeches
0,13311255,13311255


,count_star(),party
0,1549,NaN
1,6876,A
2,3474,C
3,6998540,D
4,10105,F
5,39881,I
6,1836,L
7,6464,N
8,44935,P
9,6180581,R


Rows and distinct speech ids match, no duplicates. However, the count of 13m is far fewer than the 17m rows in the total speech corpus, which means many don't have speaker data. This is fine as many of these might be procedural or other unhelpful lines, similar to the audience annotations in psets 4 and 5. This will filter them out for us for analysis. 

Party counts make sense, but we will need to replace with the actual parties here. We will also have to decide if we want to limit to D and R, or take all of them. Since the codes are not eaily designed and change, it may make sense to call it D, R and Other (data author recommendation). 

In [5]:
con.execute('''
ALTER TABLE speakermap ADD COLUMN IF NOT EXISTS party_group VARCHAR; UPDATE speakermap
SET party_group = CASE WHEN party = 'D' THEN 'Democrat'
                WHEN party = 'R' THEN 'Republican'
                ELSE 'Other'
END
''')

display(con.sql('DESCRIBE speakermap').df()[['column_name', 'column_type']])

con.sql('''
SELECT COUNT(*) as rows, party, party_group
FROM speakermap
GROUP BY party_group, party
ORDER BY rows DESC
''').df() #replacement seems to have worked properly. 

,column_name,column_type
0,speakerid,BIGINT
1,speech_id,BIGINT
2,lastname,VARCHAR
3,firstname,VARCHAR
4,chamber,VARCHAR
5,state,VARCHAR
6,gender,VARCHAR
7,party,VARCHAR
8,district,VARCHAR
9,nonvoting,VARCHAR


,rows,party,party_group
0,6998540,D,Democrat
1,6180581,R,Republican
2,44935,P,Other
3,39881,I,Other
4,16071,S,Other
5,10105,F,Other
6,6876,A,Other
7,6464,N,Other
8,3474,C,Other
9,1836,L,Other


In [6]:
con.execute(f'''
COPY speakermap TO '{DERIVED}/speakermap.parquet' (FORMAT parquet)
''')

#puts the combined file into one parquet. 

In [7]:
con.close()